In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.11 White Dwarfs and the Chandrasekhar Limit: Pauli versus Gravity

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.11",
    title="White Dwarfs and the Chandrasekhar Limit: Pauli versus Gravity",
    blurb="The pressure that made copper stiff is asked to hold up a star. For a solar "
    "mass of dead carbon it succeeds — producing planets-sized stars that shrink as "
    "they grow heavier — but the shrinking densifies the electrons, relativity "
    "softens their pressure, and above one point four solar masses gravity wins at "
    "every radius. We build the exact equation of state, certify our integrator on a "
    "case with an analytic answer, compute the limit to four digits, and land Sirius "
    "B on the resulting curve to four percent — using nothing but an ideal Fermi gas "
    "and four constants of nature.",
    difficulty="advanced",
    estimate="200–240 min",
)

## Notebook overview

This is Movement III's summit, and the keeping of the oldest promise in the quantum
half of the course: [§6.20](../06-quantum-mechanics/identical-particles.ipynb) promised that the antisymmetry postulate holds up dead stars, and this
notebook does the integral. A white dwarf is the Fermi sea of [§7.9](fermi-gas-zero-temperature.ipynb) with gravity for a container — a
solar mass of spent carbon and oxygen whose electrons, squeezed to exactly the densities where
[§7.9](fermi-gas-zero-temperature.ipynb) located the relativistic threshold, support the star with pressure made of counting.

The foundation is the **exact equation of state**. The relativistic pressure integral is
evaluated by quadrature and verified against Chandrasekhar's closed form $P = A f(x)$ to six
digits, its two limits recovered honestly (the $n^{5/3}$ of [§7.9](fermi-gas-zero-temperature.ipynb) below the threshold, the
softened $n^{4/3}$ above), and the derivative identity $f'(x) = 8x^4/\sqrt{1+x^2}$, the
workhorse of the structure equations, confirmed numerically. A four-line **scaling argument**
then exposes the doom before any differential equation: nonrelativistic degeneracy energy scales
as $1/R^2$ against gravity's $1/R$, so every mass finds an equilibrium radius (and the
mass–radius law comes out *inverted*: heavier is smaller); but ultrarelativistic degeneracy
scales as $1/R$ (the *same* power as gravity), so above the mass where the coefficients cross,
no equilibrium exists at any radius. That crossing mass is built from $\hbar$, $c$, $G$, and the
proton mass, with no astronomy in it.

The machinery is **Lane–Emden**, deployed under the course's standing discipline: certify the
integrator on a case with an analytic answer before trusting it anywhere. The $n = 1$ polytrope
must return $\xi_1 = \pi$ and $-\xi_1^2\theta'(\xi_1) = \pi$ exactly — it does, to six digits —
and only then do we harvest the classic constants for $n = 3/2$ and $n = 3$. The $n = 3$
polytrope carries the limit's mathematical fingerprint: its mass contains *no central density* —
every ultrarelativistic star weighs the same — and that unique weight is the **Chandrasekhar
mass**, $1.456\,M_\odot$ for $\mu_e = 2$, computed here to four digits. The showpiece integrates
the real thing: the structure equations with the exact EOS, swept across six decades of central
density, the mass saturating visibly at the limit while the radius collapses; and **Sirius B
lands on the computed curve to four percent**: an Earth-sized star eight and a half light-years
away, matched by an ideal gas and four constants of nature.

The payoffs close the summit. A dwarf pushed past the limit by accretion detonates as a **Type
Ia supernova**, a standard candle *because* the mass is standard, and those candles are what
revealed the accelerating universe in 1998: the number computed in this notebook calibrates the
discovery of dark energy. Chandrasekhar derived it at nineteen, on the ship to England, against
Eddington's public scorn; every white dwarf ever weighed has come in under his line. Neutron
stars (neutron degeneracy plus general relativity) are the honest horizon.

> **Conventions (this notebook).** SI throughout, CODATA/IAU constants in Setup
> ($M_\odot = 1.98892\times10^{30}$ kg). The relativity dial is $x = p_F/m_ec$; the mean
> molecular weight per electron is $\mu_e = 2$ (carbon/oxygen — one electron per two nucleons),
> with the $\mu_e^{-2}$ composition scaling of the limit stated. ODE integrations use
> `scipy.integrate.solve_ivp` with terminal `events`, stated tolerances (`rtol = 1e-10` for the
> Lane–Emden constants), and *regularized* centers — starting at exactly $r = 0$ divides by
> zero, and the Lane–Emden right-hand side uses $\max(\theta, 0)^n$ because the terminal event
> overshoots zero at machine precision and a fractional power of a negative base is a NaN
> factory. The surface event ($x = 10^{-4}$) is checked for insensitivity.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the pressure integral against the closed form at six digits with both limits recovered;
> $E(R)$ owning a minimum below the critical mass and none above; the $n = 1$ certification
> returning $(\pi, \pi)$; the classic constants $(3.6538, 2.7141)$ and $(6.8968, 2.0182)$; the
> $n = 3$ mass shedding its density dependence; $M_{\text{Ch}} = 1.456\,M_\odot$; the full
> structure saturating at $1.4485\,M_\odot$; and Sirius B's radius read off the curve within a
> few percent of measurement. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*.
>
> **Scope.** Newtonian hydrostatics with the ideal degenerate EOS — exactly the theory
> Chandrasekhar had. Finite-temperature corrections are *invoked* from [§7.10](fermi-gas-finite-temperature.ipynb) in a one-line aside
> ($T/T_F \sim 10^{-3}$ even at a $10^7$ K core); Coulomb/lattice and composition corrections
> are named with the residuals they carry; neutron stars and the TOV equation are the horizon,
> named not developed. See Chandrasekhar (1931, 1935); Shapiro & Teukolsky, *Black Holes, White
> Dwarfs and Neutron Stars* (Chs. 2–3); Kippenhahn & Weigert (polytropes); Pathria & Beale
> (Ch. 8). Cross-reference [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the promise), [§7.9](fermi-gas-zero-temperature.ipynb) (the pressure and the located threshold),
> [§7.10](fermi-gas-finite-temperature.ipynb) (the finite-$T$ license), [§7.3](statistical-toolkit.ipynb) (the counting machinery), and Volume I (the ODE
> discipline, certified again here).

## Theory in brief

### The relativistic Fermi gas

Beyond the threshold of [§7.9](fermi-gas-zero-temperature.ipynb) the dispersion is $\varepsilon(p) = \sqrt{p^2c^2 + m_e^2c^4} - m_ec^2$,
and the $T = 0$ pressure follows from the momentum flux (each mode carries velocity
$v = pc^2/E$):

```{math}
:label: eq-relativistic-eos
P = \frac{1}{3\pi^2\hbar^3}\int_0^{p_F}\frac{p^4c^2}{\sqrt{p^2c^2+m_e^2c^4}}\,dp
= A\,f(x),
\qquad
A = \frac{m_e^4c^5}{24\pi^2\hbar^3},
\quad
x = \frac{p_F}{m_ec},
```

with Chandrasekhar's closed form $f(x) = x(2x^2-3)\sqrt{1+x^2} + 3\sinh^{-1}x$ and the workhorse
identity $f'(x) = 8x^4/\sqrt{1+x^2}$. The two limits are the stiff gas of [§7.9](fermi-gas-zero-temperature.ipynb) and its softened
successor: $P \to \tfrac25 n\varepsilon_F \propto n^{5/3}$ for $x \ll 1$ and $P \to
\tfrac14 n\varepsilon_F \propto n^{4/3}$ for $x \gg 1$. The physics in one sentence: relativity
caps the velocity, so compression buys less momentum flux than it used to — the stiffness law
bends exactly where [§7.9](fermi-gas-zero-temperature.ipynb) located the threshold.

### The scaling argument: why softening is fatal

For $N$ electrons (total mass $M = N\mu_e m_u$) in radius $R$:

```{math}
:label: eq-scaling-doom
E_{\text{NR}}(R) \sim \frac{a\,N^{5/3}}{R^2} - \frac{bGM^2}{R}
\quad(\text{minimum at } R \propto M^{-1/3}),
\qquad
E_{\text{UR}}(R) \sim \frac{a'N^{4/3} - bGM^2}{R}
\quad(\text{no minimum}),
```

The nonrelativistic gas always wins somewhere: its $1/R^2$ eventually beats gravity's $1/R$, and
the equilibrium radius *shrinks* with mass — the inverted mass–radius law, with a feedback loop
built in: heavier $\to$ smaller $\to$ denser $\to$ more relativistic $\to$ softer. The
ultrarelativistic gas carries the *same* $1/R$ as gravity: the radius cancels, the numerator's
sign decides everything, and it flips at

$$
M_{\text{Ch}} \sim \frac{(\hbar c/G)^{3/2}}{(\mu_e m_H)^2} \approx 0.47\,M_\odot
\times\big(\text{an }O(1)\text{ prefactor, measured below: } 3.10\big).
$$

Pause on this object: a *stellar* mass assembled from Planck's constant, the speed of light,
Newton's constant, and the proton mass — quantum mechanics, relativity, and gravity fixing the
maximum weight of dead matter with no astronomy consulted.

### Lane–Emden, certified then deployed

Hydrostatic equilibrium plus mass continuity for a polytrope $P = K\rho^{1+1/n}$ reduces, with
$\rho = \rho_c\theta^n$ and $r = \alpha\xi$, to

```{math}
:label: eq-lane-emden
\theta'' + \frac{2}{\xi}\theta' = -\theta^n,
\qquad
\theta(0) = 1,\ \theta'(0) = 0,
\qquad
\alpha^2 = \frac{(n+1)K\rho_c^{\frac{1}{n}-1}}{4\pi G},
```

integrated to the first zero $\xi_1$ (the surface), with the two numbers that matter being
$\xi_1$ and $m_3 \equiv -\xi_1^2\theta'(\xi_1)$ (the dimensionless mass). The course's standing
discipline applies: the $n = 1$ case has the analytic solution $\theta = \sin\xi/\xi$, so the
solver must return $\xi_1 = \pi$ and $m_3 = \pi$ before it is trusted with anything else. Then:
$n = 3/2$ (the NR gas) gives $(3.6538, 2.7141)$; $n = 3$ (the UR gas) gives $(6.8968, 2.0182)$.
Mass and radius assemble as $M = 4\pi\alpha^3\rho_c\,m_3$ and $R = \alpha\,\xi_1$.

### The n = 3 signature and the mass

Carry the $\rho_c$ powers through: $M \propto \rho_c^{(3-n)/2n}$. For $n = 3/2$ this is
$\sqrt{\rho_c}$ — squeeze harder, weigh more, every mass reachable. For $n = 3$ the exponent
*vanishes*:

```{math}
:label: eq-chandrasekhar-mass
M_{n=3} = 4\pi\,m_3\left(\frac{K}{\pi G}\right)^{3/2}
\quad(\text{no }\rho_c),
\qquad
K_{\text{UR}} = \frac{(3\pi^2)^{1/3}}{4}\,\frac{\hbar c}{(\mu_e m_u)^{4/3}}
\;\Longrightarrow\;
M_{\text{Ch}} = 1.456\,M_\odot\ (\mu_e = 2).
```

Every ultrarelativistic polytrope weighs the same — and a mass independent of how hard you
squeeze is a mass nothing can exceed: past it, compression raises gravity's demand exactly as
fast as the gas's supply, forever. The $\mu_e^{-2}$ composition dependence is worth a sentence:
helium and carbon/oxygen cores share $\mu_e = 2$; an iron core's $\mu_e = 2.15$ *lowers* the
limit.

### The full structure, and Sirius B on the curve

The real star drops the polytrope and uses the exact EOS, in the $(x, m)$ variables where the
chain rule and the $f'$ identity make the system clean:

```{math}
:label: eq-full-structure
A\,f'(x)\,\frac{dx}{dr} = -\frac{Gm\rho}{r^2},
\qquad
\frac{dm}{dr} = 4\pi r^2\rho,
\qquad
\rho = \mu_e m_u\,\frac{(m_ec/\hbar)^3}{3\pi^2}\,x^3,
```

integrated from a regularized center to the surface event across six decades of central density.
The results carry the whole story: $M$ climbs from $0.11$ to $1.4485\,M_\odot$ and *saturates*
at the limit while $R$ falls from $18\,250$ to $760$ km; the low-mass slope
$d\ln R/d\ln M = -0.360$ approaches the scaling argument's $-1/3$ (the residual honestly
attributed to $x_c = 0.3$ not being deeply nonrelativistic). And the data land: **Sirius B**
($1.018\,M_\odot$, measured $R = 5.8\times10^6$ m) sits on the computed curve at
$5.58\times10^6$ m (four percent), with 40 Eridani B alongside, the residuals carried by named,
small corrections (Coulomb/lattice, composition profile, finite temperature).

### The payoffs

A maximum mass built from fundamental constants is the same number in every galaxy, and
astronomy has turned that universality into a measuring instrument; the chain of reasoning,
compressed to a schematic:

```{math}
:label: eq-payoffs
\text{accretion} \to M \nearrow M_{\text{Ch}} \to \text{thermonuclear detonation at a standard mass}
\to \text{a standard candle.}
```

A white dwarf fed by a companion approaches the limit and ignites: because the detonating mass
is *always the same*, the luminosity is too, and Type Ia supernovae become distance markers
visible across the universe. The 1998 supernova surveys built on these candles revealed the
accelerating expansion (dark energy), so the number computed in this notebook calibrates
cosmology (with the honest note that the progenitor channels, single- versus double-degenerate,
remain an active question: the candle works better than our understanding of the wick). The
history deserves its sentence: Chandrasekhar, nineteen, derived the limit aboard ship to
Cambridge in 1930; Eddington mocked it publicly for years; the 1983 Nobel and every white dwarf
ever weighed said otherwise. Past the limit lies collapse: neutron stars, where *neutron*
degeneracy plus general relativity (the TOV equation replacing Newtonian hydrostatics) set a
second, less certain limit near $2$–$3\,M_\odot$ — and past that, black holes. Named, not
developed.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad, solve_ivp

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: SI throughout; x = p_F/m_e c is the relativity dial;
# μ_e = 2 (C/O: one electron per two nucleons) with the μ_e^(-2) scaling of the limit
# stated where it matters. ODE discipline: solve_ivp with terminal events, stated
# tolerances, REGULARIZED centers (r = 0 divides by zero), and max(θ, 0)^n inside
# Lane–Emden (the event overshoots zero at machine precision; a fractional power of a
# negative base is a NaN factory).
HBAR = 1.054571817e-34  # J s
M_E = 9.1093837015e-31  # kg
C_LIGHT = 2.99792458e8  # m/s (exact)
G_NEWTON = 6.67430e-11  # m^3 kg^-1 s^-2
M_U = 1.66053906660e-27  # kg
M_SUN = 1.98892e30  # kg (IAU)
MU_E = 2.0  # mean molecular weight per electron (C/O)

A_EOS = M_E**4 * C_LIGHT**5 / (24.0 * np.pi**2 * HBAR**3)  # ≈ 6.00e21 Pa
RHO_X = MU_E * M_U * (M_E * C_LIGHT / HBAR) ** 3 / (3.0 * np.pi**2)  # ρ = RHO_X · x^3
N_X = (M_E * C_LIGHT / HBAR) ** 3 / (3.0 * np.pi**2)  # n_e = N_X · x^3


def f_chandra(x):
    """Chandrasekhar's pressure function f(x) = x(2x^2 − 3)√(1+x^2) + 3 asinh(x).

    The exact antiderivative of the relativistic pressure integral in the dial
    x = p_F/m_e c: P = A f(x) with A = m_e^4 c^5/24π^2 ħ^3 (eq-relativistic-eos).
    Limits worth knowing by heart: f → (8/5)x^5 as x → 0 (the n^(5/3) gas of §7.9)
    and f → 2x^4 as x → ∞ (the softened n^(4/3) gas of this notebook's summit).

    Parameters
    ----------
    x : float or numpy.ndarray
        The relativity parameter p_F/m_e c.

    Returns
    -------
    float or numpy.ndarray
        f(x), dimensionless.
    """
    return x * (2.0 * x**2 - 3.0) * np.sqrt(1.0 + x**2) + 3.0 * np.arcsinh(x)


def fprime(x):
    """The workhorse identity f'(x) = 8x^4/√(1+x^2), verified in Exercise 1.

    This derivative is what makes the (x, m) formulation of the structure equations
    clean: dP/dr = A f'(x) dx/dr, so hydrostatic equilibrium becomes a first-order
    equation in x directly — no numerical differentiation of the EOS anywhere in the
    star.

    Parameters
    ----------
    x : float or numpy.ndarray
        The relativity parameter.

    Returns
    -------
    float or numpy.ndarray
        f'(x).
    """
    return 8.0 * x**4 / np.sqrt(1.0 + x**2)


def pressure_quad(x):
    """The relativistic T = 0 pressure by direct quadrature of the momentum-flux integral.

    P = (1/3π^2ħ^3) ∫_0^{p_F} p^4 c^2/√(p^2c^2 + m^2c^4) dp — each mode contributes
    momentum p at velocity v = pc^2/E, and the isotropic flux average supplies the
    1/3. This is the definitional route; Exercise 1 gates it against the closed form
    A f(x) at six digits, after which the closed form does all the work.

    Parameters
    ----------
    x : float
        The relativity parameter p_F/m_e c.

    Returns
    -------
    float
        Pressure in Pa.
    """
    p_F = x * M_E * C_LIGHT
    integrand = (
        lambda p: p**4 * C_LIGHT**2 / np.sqrt(p**2 * C_LIGHT**2 + M_E**2 * C_LIGHT**4)
    )
    return quad(integrand, 0.0, p_F, limit=200)[0] / (3.0 * np.pi**2 * HBAR**3)


def lane_emden(n, rtol=1e-10):
    """Integrate the Lane–Emden equation to the surface: returns (ξ_1, −ξ_1^2 θ'(ξ_1)).

    θ'' + (2/ξ)θ' = −max(θ, 0)^n with the regularized series start θ = 1 − ξ^2/6,
    θ' = −ξ/3 at ξ_0 = 1e-6 (starting at exactly zero divides by zero), a terminal
    event at θ = 0, and rtol = 1e-10 (the classic four-digit constants need it). The
    max(θ, 0) guards the RHS against the event's machine-precision overshoot — a
    fractional n on a negative θ is the notebook's stated NaN factory. Certified on
    the analytic n = 1 case (ξ_1 = π, m_3 = π) before deployment.

    Parameters
    ----------
    n : float
        The polytropic index.
    rtol : float, optional
        solve_ivp relative tolerance (default 1e-10; atol 1e-12).

    Returns
    -------
    tuple of float
        (ξ_1, m_3) with m_3 = −ξ_1^2 θ'(ξ_1).
    """

    def rhs(xi, y):
        theta, dtheta = y
        return [dtheta, -max(theta, 0.0) ** n - 2.0 * dtheta / xi]

    surface = lambda xi, y: y[0]
    surface.terminal = True
    surface.direction = -1
    xi0 = 1e-6
    y0 = [1.0 - xi0**2 / 6.0, -xi0 / 3.0]
    sol = solve_ivp(rhs, (xi0, 50.0), y0, events=surface, rtol=rtol, atol=1e-12)
    xi1 = float(sol.t_events[0][0])
    dtheta1 = float(sol.y_events[0][0][1])
    return xi1, -(xi1**2) * dtheta1


def integrate_star(x_c, surface_x=1e-4, rtol=1e-8):
    """The full white-dwarf structure with the exact EOS, in the (x, m) variables.

    A f'(x) dx/dr = −G m ρ/r^2 and dm/dr = 4π r^2 ρ with ρ = RHO_X·x^3
    (eq-full-structure) — hydrostatics with no polytropic approximation anywhere.
    The center is regularized at r_0 = 1 m with the uniform-density mass
    m_0 = (4π/3)ρ_c r_0^3; the surface is a terminal event at x = surface_x, small
    enough not to bias the radius (Exercise 5 shows the insensitivity once) yet large
    enough to terminate before the dx/dr ∝ 1/x^4-driven steepening stalls the step
    size.

    Parameters
    ----------
    x_c : float
        Central relativity parameter (ρ_c = RHO_X · x_c^3).
    surface_x : float, optional
        The surface event threshold (default 1e-4).
    rtol : float, optional
        solve_ivp relative tolerance (default 1e-8).

    Returns
    -------
    tuple of float
        (R in metres, M in kg).
    """
    rho_c = RHO_X * x_c**3

    def rhs(r, y):
        x, m = y
        x_safe = max(x, 1e-12)
        rho = RHO_X * x_safe**3
        dxdr = -G_NEWTON * m * rho / (r**2 * A_EOS * fprime(x_safe))
        return [dxdr, 4.0 * np.pi * r**2 * rho]

    surface = lambda r, y: y[0] - surface_x
    surface.terminal = True
    surface.direction = -1
    r0 = 1.0
    y0 = [x_c, 4.0 * np.pi / 3.0 * rho_c * r0**3]
    sol = solve_ivp(rhs, (r0, 1e9), y0, events=surface, rtol=rtol, atol=[1e-12, 1e10])
    return float(sol.t_events[0][0]), float(sol.y_events[0][0][1])


def chandrasekhar_mass(mu_e=2.0):
    """The Chandrasekhar mass M_Ch = 4π m_3 (K_UR/πG)^(3/2), in kg (eq-chandrasekhar-mass).

    K_UR = (3π^2)^(1/3)(ħc/4)/(μ_e m_u)^(4/3) is the ultrarelativistic polytropic
    constant, read off the x → ∞ limit of the exact EOS; m_3 = 2.01824 is the n = 3
    Lane–Emden mass computed (and certified) in Exercise 3. The μ_e^(-2) scaling is
    explicit: iron cores (μ_e = 2.15) carry a LOWER limit than C/O.

    Parameters
    ----------
    mu_e : float, optional
        Mean molecular weight per electron (default 2.0).

    Returns
    -------
    float
        The limiting mass in kg.
    """
    K_UR = (
        (3.0 * np.pi**2) ** (1.0 / 3.0)
        * HBAR
        * C_LIGHT
        / (4.0 * (mu_e * M_U) ** (4.0 / 3.0))
    )
    _, m3 = lane_emden(3.0)
    return 4.0 * np.pi * m3 * (K_UR / (np.pi * G_NEWTON)) ** 1.5

## Exercise 1 — The exact equation of state

The pressure of a Fermi gas at any speed — integral, closed form, and the two slopes it
interpolates. Cite {eq}`eq-relativistic-eos`.

1. Derive the pressure integral $P = (1/3\pi^2\hbar^3)\int_0^{p_F}p^4c^2/E(p)\,dp$ for the
   relativistic dispersion.
2. Use `pressure_quad` and `pressure_closed` ($A\,f(x)$) and verify agreement to at least six
   digits at $x = 0.1, 1, 5$.
3. Recover the limits — $P/[\tfrac25 n\varepsilon_F] = 0.9991$ at $x = 0.05$ and
   $P/[n\varepsilon_F/4] = 0.9996$ at $x = 50$ — and plot $P(n)$ on log–log to read off the
   $5/3 \to 4/3$ bend at $x \sim 1$.
4. Verify the identity $f'(x) = 8x^4/\sqrt{1+x^2}$ by central differences on $f$ (the
   workhorse of Exercise 5),
   and state the physics of the softening: relativity caps velocity, so compression buys less
   pressure. (Computation + prose.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    P_ratios,
    [1.0, 1.0, 1.0],
    "the relativistic EOS: the pressure integral equals Chandrasekhar's closed form",
    rtol=1e-6,
)
validate.close(
    [ratio_nr, ratio_ur],
    [0.9991, 0.9996],
    "both limits recovered: the (2/5)nε_F of §7.9 below the threshold, the softened nε_F/4 above",
    rtol=1e-3,
)
validate.check(
    fp_err < 1e-8,
    "the workhorse identity f'(x) = 8x^4/√(1+x^2), confirmed before Exercise 5 leans on it",
    f"worst deviation {fp_err:.1e}",
)

## Exercise 2 — Four lines of doom

The scaling argument: why a softened pressure cannot hold arbitrarily heavy stars. Cite
{eq}`eq-scaling-doom`.

1. Assemble $E(R) = aN^{5/3}/R^2 - bGM^2/R$ for the NR gas (coefficients from the
   $E = \tfrac35N\varepsilon_F$ of [§7.9](fermi-gas-zero-temperature.ipynb) and the uniform sphere's $b = 3/5$), minimize, and obtain
   $R \propto M^{-1/3}$ — the inverted mass–radius law, with the feedback loop stated.
2. Repeat for the UR gas: $E(R) = (a'N^{4/3} - bGM^2)/R$ — no minimum; the numerator's sign
   decides.
3. Solve the sign change for $M$ and exhibit $M_{\text{Ch}} \sim (\hbar c/G)^{3/2}/(\mu_em_H)^2$;
   evaluate the dimensional combination ($\approx 0.47\,M_\odot$ before the prefactor) and
   reflect on its contents: $\hbar$, $c$, $G$, $m_H$ — no astronomy.
4. Plot $E(R)$ for masses below, at, and above critical: minimum, marginal flatness, monotonic
   collapse. (Computation + prose.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    minima["below"] and not minima["above"],
    "the scaling argument: an equilibrium radius exists below the critical mass and not above",
)
validate.close(
    M_dimensional / M_SUN,
    0.470,
    "(ħc/G)^(3/2)/(μ_e m_u)^2 ≈ 0.47 M_sun: a stellar mass with no astronomy in it",
    rtol=1e-2,
)

## Exercise 3 — Lane–Emden, certified on an exact case

The stellar ODE, and the course's discipline: never trust an integrator you haven't tested
against an analytic answer. Cite {eq}`eq-lane-emden`.

1. Derive the Lane–Emden reduction from hydrostatic equilibrium + mass continuity for
   $P = K\rho^{1+1/n}$.
2. Use `lane_emden(n)` (`scipy.integrate.solve_ivp`, terminal event $\theta = 0$,
   `rtol = 1e-10`, regularized series start, $\max(\theta,0)^n$ in the RHS — the NaN trap
   stated).
3. Certify on $n = 1$: confirm $\xi_1 = \pi$ and $-\xi_1^2\theta'(\xi_1) = \pi$ to at least six
   digits against the analytic $\theta = \sin\xi/\xi$.
4. Produce the working constants — $n = 3/2 \to (3.6538, 2.7141)$; $n = 3 \to (6.8968,
   2.0182)$ — and plot the three profiles.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [xi1_n1, m3_n1],
    [np.pi, np.pi],
    "Lane–Emden certified on the analytic n = 1 case: (π, π) before anything is trusted",
    rtol=1e-6,
)
validate.close(
    [xi1_32, m3_32, xi1_3, m3_3],
    [3.6538, 2.7141, 6.8968, 2.0182],
    "the classic constants: n = 3/2 and n = 3, to the digits the literature quotes",
    rtol=1e-4,
)

## Exercise 4 — The mass that forgot its density

The $n = 3$ polytrope's signature, and the number it fixes. Cite {eq}`eq-chandrasekhar-mass`.

1. Assemble $M(\rho_c) = 4\pi\alpha^3\rho_c m_3$ from the Lane–Emden outputs and demonstrate
   numerically: $M \propto \sqrt{\rho_c}$ for $n = 3/2$ but $M$ *constant* for $n = 3$ — the
   exponent $(3-n)/2n$ vanishing.
2. Derive $K_{\text{UR}} = (3\pi^2)^{1/3}(\hbar c/4)/(\mu_em_u)^{4/3}$ from the UR limit of
   Exercise 1's EOS.
3. Compute $M_{\text{Ch}} = 4\pi m_3(K_{\text{UR}}/\pi G)^{3/2} = 1.456\,M_\odot$ for
   $\mu_e = 2$ (the `chandrasekhar_mass` helper), and extract the scaling prefactor
   $c_0 = M_{\text{Ch}}(\mu_em_u)^2/(\hbar c/G)^{3/2}$.
4. Read the fingerprint (prose): a mass independent of central density is a mass no compression
   can rescue — the limit is not where the star breaks but where equilibrium ceases to exist;
   note the $\mu_e^{-2}$ composition dependence.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    M_Ch_solar,
    1.456,
    "the Chandrasekhar mass: 1.456 M_sun for μ_e = 2, from ħ, c, G, m_u and one ODE constant",
    rtol=1e-3,
)
validate.check(
    abs(exponent_32 - 0.5) < 1e-3 and abs(exponent_3) < 1e-6,
    "the n = 3 signature: M ∝ √ρ_c for the NR polytrope, M independent of ρ_c for the UR one",
    f"exponents {exponent_32:.4f}, {exponent_3:.1e}",
)
validate.close(
    c0_prefactor,
    3.098,
    "the scaling argument's O(1) prefactor, measured: c_0 = 3.098",
    rtol=1e-2,
)

## Exercise 5 — The real star, integrated

The exact EOS in the structure equations, swept across six decades of central density — the
summit computation. Cite {eq}`eq-full-structure`.

1. Formulate the $(x, m)$ system — $A f'(x)\,dx/dr = -Gm\rho/r^2$, $dm/dr = 4\pi r^2\rho$,
   $\rho = \mu_em_u(m_ec/\hbar)^3x^3/3\pi^2$ — and use `integrate_star` (`solve_ivp`, surface
   event $x = 10^{-4}$ with an insensitivity check, regularized center).
2. Sweep $x_c = 0.3 \to 32$ (log-spaced $\rho_c = 5\times10^7 \to 6\times10^{13}$ kg/m³) and
   tabulate $(\rho_c, R, M)$: the mass climbing $0.1098 \to 1.4485\,M_\odot$ and saturating
   toward $1.456$, the radius falling $18\,250 \to 760$ km.
3. Verify the low-mass slope $d\ln R/d\ln M = -0.360$ and attribute the residual from $-1/3$
   honestly ($x_c = 0.3$ is not yet deeply nonrelativistic).
4. Plot $M(\rho_c)$ with the $M_{\text{Ch}}$ asymptote and $R(M)$ — the movement's summit
   figures.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    M_solar[-1],
    1.4485,
    "the full structure saturates at the Chandrasekhar mass (1.4485 at ρ_c = 6e13 kg/m³)",
    rtol=1e-2,
)
validate.close(
    [M_solar[0], R_sweep[0] / 1e3, R_sweep[-1] / 1e3],
    [0.1098, 18250.0, 760.0],
    "the sweep's corners: 0.11 → 1.45 M_sun while R falls 18 250 → 760 km",
    rtol=2e-2,
)
validate.check(
    -0.42 < slope_low < -0.34 and abs(R_a - R_b) / R_a < 1e-4,
    "the low-mass slope near −1/3 (softened honestly by x_c = 0.3), on an event-insensitive R",
    f"slope {slope_low:.3f}",
)

## Exercise 6 — Sirius B on the curve

A real star in the sky, placed on a curve made of counting. Cite {eq}`eq-full-structure`.

1. Read the curve at Sirius B's measured mass ($1.018\,M_\odot$) with `numpy.interp`
   (monotonicity of the $(M, R)$ table checked): the computed radius.
2. Compare with the measured $5.8\times10^6$ m, and place 40 Eridani B ($0.573\,M_\odot$,
   $\sim9.7\times10^6$ m) on the same plot.
3. Name the residual's physics (Coulomb/lattice corrections, composition profile, finite
   temperature — each small, each cited) and verify the finite-$T$ claim with the tools of [§7.10](fermi-gas-finite-temperature.ipynb):
   $T/T_F \sim 10^{-3}$ even at a $10^7$ K core (one-line computation).
4. Weigh the result (prose): an ideal Fermi gas, four constants of nature, and an Earth-sized
   star eight and a half light-years away agreeing to a few percent — the volume's strongest
   single validation.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    R_sirius_computed,
    5.58e6,
    "Sirius B lands on the degenerate curve (5.58e6 m computed vs 5.8e6 m measured: 4%)",
    rtol=2e-2,
)
validate.check(
    mono_ok and abs(R_sirius_computed - R_sirius_measured) / R_sirius_measured < 0.06,
    "a real star, an ideal gas, four constants: agreement at the few-percent level",
    f"residual {abs(R_sirius_computed - R_sirius_measured) / R_sirius_measured:.1%}",
)
validate.check(
    T_over_TF < 5e-3,
    "the license of §7.10 holds at stellar cores: T/T_F ~ 1e-3 even at 1e7 K",
    f"T/T_F = {T_over_TF:.1e}",
)

## Exercise 7 — The candle that measured the universe

The limit's modern payoff, computed at the level it deserves. Cite {eq}`eq-payoffs`.

1. State the Type Ia mechanism (accretion toward $M_{\text{Ch}} \to$ thermonuclear detonation at
   a standard mass) and why standard mass $\Rightarrow$ standard luminosity $\Rightarrow$
   distance ladder.
2. Compute the available nuclear energy of a Chandrasekhar mass of carbon burning to iron-group
   ($\Delta\varepsilon \approx 1.2\times10^{-3}c^2$ per unit mass from the binding-energy
   difference; assumptions stated) and compare with the star's *net* binding energy — enough to
   unbind it several times over.
3. Estimate the peak-luminosity scale from $\sim0.6\,M_\odot$ of $^{56}$Ni decay powering the
   light curve (stated half-life bookkeeping; order of magnitude only).
4. Close the loop (prose): the 1998 supernova surveys that revealed the accelerating universe
   were calibrated on this notebook's number — with the honest note that the progenitor
   channels remain an active question.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    E_nuclear / E_binding_net > 3.0,
    "why the standard mass detonates: nuclear energy exceeds the net binding severalfold",
    f"ratio {E_nuclear / E_binding_net:.1f}×",
)
validate.check(
    1e35 < L_peak < 1e38,
    "the candle's wattage: Ni-56 decay powers a ~1e36 W light curve — galaxy-scale, briefly",
    f"L ~ {L_peak:.1e} W",
)

## Exercise 8 — The summit

The movement began with a pressure made of counting and ends with that pressure weighed against
gravity on the largest scale matter offers. For most stars the ledger balances: a solar mass of
spent carbon settles into an Earth-sized sphere, and — alone among the objects of astronomy —
grows *smaller* as it grows heavier, because each added gram is accommodated only by higher density. But
density is the one currency that devalues the pressure itself: the electrons go relativistic,
the stiffness law bends from five-thirds to four-thirds, and at that exponent gravity and
degeneracy scale identically — the contest stops depending on radius and starts depending only
on mass. Above 1.456 solar masses of dead matter, arithmetic votes for collapse. We certified
every tool against an exact answer, computed the limit to four digits, and found a real star
sitting on our curve to four percent. The Pauli principle, which in [§6.20](../06-quantum-mechanics/identical-particles.ipynb) was a symmetry
postulate and in [§7.9](fermi-gas-zero-temperature.ipynb) the stiffness of potassium, here decides which stars are allowed to die
quietly — and its failure, past the limit, lights the candles by which cosmology measured the
accelerating universe.

Chandrasekhar was nineteen, on a ship, when he noticed that $\hbar$, $c$, $G$ and the proton
mass assemble into a stellar weight. Eddington mocked the result publicly for years. Every white
dwarf ever weighed has come in under the line. It is one of the cleanest verdicts in the history
of the subject: the universe sided with the student who did the integral.

Next the movement returns to Earth with a confession: the same free sea that held up Sirius B
cannot explain why diamond does not conduct — the gas needs a lattice ([§7.12](bloch-theorem-band-structure.ipynb)).

## Notebook summary

Movement III's summit: Pauli versus gravity, decided by an exponent.

- **The exact EOS** {eq}`eq-relativistic-eos`: the pressure integral meets Chandrasekhar's
  $Af(x)$ at six digits; both limits recovered ($0.9991$, $0.9996$); the workhorse
  $f'(x) = 8x^4/\sqrt{1+x^2}$ certified — the $5/3 \to 4/3$ bend is a computed fact.
- **Four lines of doom** {eq}`eq-scaling-doom`: NR degeneracy's $1/R^2$ guarantees every mass an
  equilibrium (at $R \propto M^{-1/3}$: heavier is smaller); UR degeneracy's $1/R$ matches
  gravity, the radius cancels, and the sign flips at $\sim(\hbar c/G)^{3/2}/(\mu_em_H)^2$ —
  $0.47\,M_\odot$ of pure constants, no astronomy.
- **Lane–Emden, certified** {eq}`eq-lane-emden`: the $n = 1$ analytic case returns $(\pi, \pi)$
  to six digits *before* the solver is trusted; then the classic $(3.6538, 2.7141)$ and
  $(6.8968, 2.0182)$.
- **The mass that forgot its density** {eq}`eq-chandrasekhar-mass`: the $n = 3$ exponent
  $(3-n)/2n$ vanishes (measured: $10^{-16}$ against $n = 3/2$'s exact $0.5$), and with
  $K_{\text{UR}}$ from the EOS: $M_{\text{Ch}} = 1.456\,M_\odot$, prefactor $c_0 = 3.098$;
  $\mu_e^{-2}$ composition scaling noted.
- **The real star** {eq}`eq-full-structure`: the $(x, m)$ system with the exact EOS, swept
  across six decades — $M: 0.1098 \to 1.4485\,M_\odot$ saturating at the limit, $R: 18\,250 \to
  760$ km, the low-mass slope $-0.360$ honestly attributed; and **Sirius B on the curve to
  4%** (40 Eridani B alongside), with the residual's physics and the finite-$T$ license of [§7.10](fermi-gas-finite-temperature.ipynb)
  ($T/T_F \sim 10^{-3}$) named.
- **The payoffs** {eq}`eq-payoffs`: a standard mass makes a standard candle — the nuclear budget
  ($3\times10^{44}$ J against a $5\times10^{43}$ J net binding) detonates the star, $0.6\,
  M_\odot$ of $^{56}$Ni lights a $10^{36}$ W curve, and the 1998 cosmology built on it; the
  history (the boat, Eddington, 1983) and the horizon (neutron stars, TOV, $2$–$3\,M_\odot$)
  told straight.

A pressure made of counting, weighed against gravity — and the number that came out calibrates
the universe's acceleration.

## Outlook

- **Back to Earth ([§7.12](bloch-theorem-band-structure.ipynb), [§7.13](semiconductor-statistics.ipynb)).** The sea meets a lattice: Bloch, bands, and why insulators
  exist; then Fermi–Dirac in a gap.
- **Past the limit.** Neutron stars — neutron degeneracy plus general relativity (TOV), the
  $2$–$3\,M_\odot$ second limit — and black holes beyond: horizons, named.
- **The candle's open questions.** Progenitor channels and light-curve standardization: the
  cosmology works better than the wick is understood.
- **White-dwarf refinements.** Coulomb/lattice corrections, crystallizing cores, pulsating
  dwarfs — the percent-level physics carrying Sirius B's residual: horizons, named.
- **Cross-reference** [§6.20](../06-quantum-mechanics/identical-particles.ipynb) (the promise, kept in full), [§7.9](fermi-gas-zero-temperature.ipynb) (the pressure; the threshold this
  notebook detonated), [§7.10](fermi-gas-finite-temperature.ipynb) (the finite-$T$ license, still parts-per-thousand at $10^7$ K), [§7.3](statistical-toolkit.ipynb)
  (the counting machinery), Volume I (the ODE discipline, certified again).

In [ ]:
from ecp.style import footer

footer()